In [ ]:
# ── Cell 1: Setup ────────────────────────────────────────────────────────
import sys
sys.path.append('..')

import torch
import numpy as np
import random
import copy
import segmentation_models_pytorch as smp
from sklearn.model_selection import train_test_split
import os

from configs.config import Config
from data.dataset import BraTSDataset
from models.bu_net import BUNetMultiTask

# Added the sampler imports!
from data.few_shot_sampler import FewShotSampler, kshot_finetune_eval 

random.seed(42)
np.random.seed(42)
torch.manual_seed(42)

print(f"✓ Device: {Config.DEVICE}")

In [ ]:
# ── Cell 2: Load data (FIXED SPLITS) ─────────────────────────────────────
def get_patient_ids(data_path):
    dirs = [f.path for f in os.scandir(data_path) if f.is_dir()]
    return [d[d.rfind('/')+1:] for d in dirs]

all_ids = get_patient_ids(Config.TRAIN_DATASET_PATH)

# FIX: Use the exact same splits as your training notebook to prevent data leaks!
train_test_ids, val_ids = train_test_split(all_ids, test_size=0.2, random_state=42)
train_ids, test_ids = train_test_split(train_test_ids, test_size=0.125, random_state=42)

train_dataset = BraTSDataset(train_ids, Config.TRAIN_DATASET_PATH)
val_dataset = BraTSDataset(val_ids, Config.TRAIN_DATASET_PATH)
test_dataset = BraTSDataset(test_ids, Config.TRAIN_DATASET_PATH)

print(f"✓ Splits matched -> Train: {len(train_ids)}, Val: {len(val_ids)}, Test: {len(test_ids)}")
print(f"✓ Datasets loaded")

In [ ]:
# Cell 3: Load baseline model
baseline_model = BUNetMultiTask(
    encoder_name='resnet34',
    in_channels=2,
    num_classes=4,
    num_tumor_types=5
).to(Config.DEVICE)

checkpoint = torch.load('./checkpoints/best_model.pth', weights_only=False)
baseline_model.load_state_dict(checkpoint['model_state_dict'])

print("✓ Baseline model loaded")

In [ ]:
# Cell 4: Define baseline_finetune_k_shot function
def baseline_finetune_k_shot(base_model, train_dataset, val_dataset, k=20):
    """Baseline: fine-tune on k examples"""
    model = copy.deepcopy(base_model)
    optimizer = torch.optim.Adam(model.parameters(), lr=0.0001)
    loss_fn = smp.losses.DiceLoss(mode='multiclass')
    
    # Sample k examples
    indices = random.sample(range(len(train_dataset)), k)
    
    # Fine-tune
    model.train()
    for epoch in range(20):
        for idx in indices:
            sample = train_dataset[idx]
            img = sample['image'].unsqueeze(0).to(Config.DEVICE)
            mask = sample['mask'].unsqueeze(0).to(Config.DEVICE)
            
            optimizer.zero_grad()
            seg_out, _ = model(img)
            loss = loss_fn(seg_out, mask)
            loss.backward()
            optimizer.step()
    
    # Evaluate
    model.eval()
    val_indices = random.sample(range(len(val_dataset)), min(20, len(val_dataset)))
    total_dice = 0
    
    with torch.no_grad():
        for idx in val_indices:
            sample = val_dataset[idx]
            img = sample['image'].unsqueeze(0).to(Config.DEVICE)
            mask = sample['mask'].unsqueeze(0).to(Config.DEVICE)
            
            pred, _ = model(img)
            dice_loss = loss_fn(pred, mask)
            total_dice += (1 - dice_loss.item())
    
    return total_dice / len(val_indices)

print("✓ Function defined")

In [ ]:
# Cell 5: Run k=20
print("Running baseline k=20...")
dice_k20 = baseline_finetune_k_shot(baseline_model, train_dataset, val_dataset, k=20)

print(f"\n{'='*50}")
print(f"BASELINE K=20 RESULT:")
print(f"{'='*50}")
print(f"DICE = {dice_k20:.4f}")
print(f"{'='*50}")

print(f"\nComplete baseline k-shot:")
print(f"  k=1:  0.730")
print(f"  k=5:  0.733")
print(f"  k=10: 0.714")
print(f"  k=20: {dice_k20:.4f}")